# Reinforcement Learning

# 5. Gradient Methods

This notebook presents gradient methods, useful for learning in some environment with a large state space.

We use a neural network to approximate the value function (value gradient) or the policy (policy gradient).

Please read the instructions:
* Write concise code and text.
* Check that your notebook runs without errors and in reasonable time (< 5 min).
* Clear all cell outputs before upload on Moodle.

In [ ]:
import numpy as np

In [ ]:
from model import TicTacToe, Nim, ConnectFour
from agent import Agent, OnlineEvaluation

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim

## Value gradient

We start with value-based methods. The neural network is a regressor that approximates the value function. Note that the model is supposed to be known, so we work with the value function.

In [ ]:
class Regressor(nn.Module):
    """Neural network for value function approximation. Return the value of each state.
        
    Parameters
    ----------
    environment : object of class Environment
        Environment of the agent.
    layers : list of int
        Dimensions of the hidden layers.
    """
    def __init__(self, environment, layers):
        if not hasattr(environment, 'one_hot_encode'):
            raise ValueError("The environment must have a one-hot encoding of states.")   
        super(Regressor, self).__init__()
        self.environment = environment
        state = environment.init_state()
        code = environment.one_hot_encode(state)
        input_size = len(code)
        self.nn = nn.Sequential()
        for number, size in enumerate(layers):
            output_size = size
            self.nn.add_module(f"Linear {number}", nn.Linear(input_size, output_size))
            self.nn.add_module(f"Activation {number}", nn.ReLU())
            input_size = size
        self.nn.add_module("Output", nn.Linear(input_size, 1))

    def forward(self, code):
        """Forward pass."""
        return self.nn(code)

In [ ]:
game = TicTacToe()

In [ ]:
regressor = Regressor(environment=game, layers=[10, 5])

In [ ]:
regressor.nn

In [ ]:
state = game.state
# one-hot encoding
code = game.one_hot_encode(state)
# tensor
code = torch.tensor(code).float()

In [ ]:
# value before training
value = regressor.forward(code).detach()
print(value)

## To do

* Complete the method ``get_best_actions`` of the class ``ValueGradient``. 
* Test the agent on TicTacToe, against (1) a random adversary and (2) a perfect adversary.<br>What is the first move of the agent in each case?
* Compare your results to another learning strategy (e.g., Monte-Carlo learning) and comment the results.

In [ ]:
class ValueGradient(OnlineEvaluation):
    """Agent learning by value gradient. The model is supposed to be known.
    
    Parameters
    ----------
    environment : object of class Environment
        Environment of the agent.
    player : int
        Player for games (1 or -1, default = default player of the game).
    gamma : float
        Discount rate.
    layers : list of int
        Dimensions of the hidden layers.
    """
    def __init__(self, environment, player=None, gamma=1, layers=[10, 5]):
        super(ValueGradient, self).__init__(environment=environment, player=player)  
        if not hasattr(environment, "get_next_state"):
            raise ValueError("The model must be known, with a 'get_next_state' method.")
        self.nn = Regressor(environment, layers)
        self.gamma = gamma
        
    def get_best_actions(self, state):
        """Get the best actions in some state according to the value function.""" 
        actions = self.get_actions(state)
        if len(actions) > 1:
            action_value = {}
            for action in actions:
                next_state = self.environment.get_next_state(state, action)
                reward = self.environment.get_reward(next_state)
                value = reward
                if not self.environment.is_terminal(next_state):
                    code = self.environment.one_hot_encode(next_state)
                    code = torch.tensor(code).float().unsqueeze(0)  # shape (1, input_size)
                    with torch.no_grad():
                        next_value = self.nn.forward(code).item()
                    value += self.gamma * next_value
                action_value[action] = value

            if self.player == 1:
                best_value = max(action_value.values())
            else:
                best_value = min(action_value.values())
            actions = [action for action in actions if action_value[action] == best_value]
        return actions        
    
    def get_samples(self, state, horizon, epsilon):
        """Get samples from one episode under the epsilon-greedy policy.
        
        Parameters
        ----------
        state : 
            Initial state of the episode.
        horizon : int
            Time horizon of the episode.
        epsilon : float
            Exploration rate.
        """
        self.policy = self.randomize_policy(epsilon=epsilon)
        self.environment.reset(state)
        state = self.environment.state
        states = []
        rewards = []
        for t in range(horizon):
            action = self.get_action(state)
            reward, stop = self.environment.step(action)
            states.append(state)
            rewards.append(reward)
            if stop:
                break
            state = self.environment.state
        samples = []
        sample = 0
        for reward in reversed(rewards):
            sample = reward + self.gamma * sample
            samples.append(sample)
        return reversed(states), samples
        
    def train(self, state=None, horizon=100, n_episodes=1000, batch_size=10, learning_rate=0.01, epsilon=0.1):
        """Train the neural network.
        
        Parameters
        ----------
        state : 
            Initial state of the episode.
        horizon : int
            Time horizon of the episode.
        n_episodes : int
            Number of episodes.
        batch_size : int
            Batch size for gradient descent.
        epsilon : float
            Exploration rate.        
        """
        optimizer = optim.Adam(self.nn.parameters(), lr=learning_rate)
        mse = nn.MSELoss()
        loss = 0
        for t in range(n_episodes):
            states, samples = self.get_samples(state, horizon, epsilon)
            codes = [self.environment.one_hot_encode(state) for state in states]
            codes = np.array(codes)
            codes = torch.tensor(codes).float()
            values = self.nn.forward(codes)
            samples = torch.tensor(samples).float().reshape(-1, 1)
            loss += mse(values, samples)
            if not t % batch_size:
                loss /= batch_size
                optimizer.zero_grad()
                loss.backward()
                optimizer.step()
                loss = 0


In [ ]:
game = TicTacToe()
agent = ValueGradient(game)

In [ ]:
returns = agent.get_returns()
np.unique(returns, return_counts=True)

In [ ]:
agent.train()
agent.update_policy()

In [ ]:
returns = agent.get_returns()
np.unique(returns, return_counts=True)

In [ ]:
# against "random" adversary (1)
game_random = TicTacToe(adversary_policy='random', player=1, play_first=True)
agent_random = ValueGradient(game_random)
agent_random.train()
agent_random.update_policy()

s0 = game_random.init_state()
first_move_random = agent_random.get_action(s0) 
print("First move vs random =", first_move_random)

# against "perfect" adversary (2)
game_perf = TicTacToe(adversary_policy='one_step', player=1, play_first=True)
agent_perf = ValueGradient(game_perf)
agent_perf.train()
agent_perf.update_policy()

s0 = game_perf.init_state()
first_move_perf = agent_perf.get_action(s0)
print("First move vs perfect(one_step) =", first_move_perf)

ANSWER:

Value gradient learns a value function with a neural network, so it can generalize to unseen states. Monte-Carlo is simpler but needs more samples and does not generalize as well. Both improve performance against a random opponent, but value gradient can change slightly between runs.

## Policy gradient

We now consider a policy-based method. The neural network is a classifier that returns the probability of each action.

In [ ]:
class Classifier(nn.Module):
    """Neural network for policy gradient. Return the distribution of actions in each state.
    
    Parameters
    ----------
    environment : object of class Environment
        Environment of the agent.
    layers : list of int
        Dimensions of the hidden layers.
    """
    def __init__(self, environment, layers):
        if not hasattr(environment, 'one_hot_encode'):
            raise ValueError("The environment must have a one-hot encoding of states.")   
        super(Classifier, self).__init__()
        self.environment = environment
        actions = environment.get_all_actions()
        if self.environment.is_game():
            # remove pass action
            actions = [action for action in actions if action != "pass"]
        self.actions = actions
        state = environment.init_state()
        code = environment.one_hot_encode(state)
        input_size = len(code)
        output_size = len(actions)
        self.nn = nn.Sequential()
        for number, size in enumerate(layers):
            output_size = size
            self.nn.add_module(f"Linear {number}", nn.Linear(input_size, output_size))
            self.nn.add_module(f"Activation {number}", nn.ReLU())
            input_size = size
        self.nn.add_module("Output", nn.Linear(input_size, len(actions)))
        self.nn.add_module("Softmax", nn.Softmax(dim=0))

    def forward(self, code):
        """Forward pass."""
        return self.nn(code)

In [ ]:
game = TicTacToe()

In [ ]:
classifier = Classifier(environment=game, layers=[10, 5])

In [ ]:
classifier.nn

In [ ]:
state = game.state
# one-hot encoding
code = game.one_hot_encode(state)
# tensor
code = torch.tensor(code).float()

In [ ]:
# probability of each action before training
probs = classifier.forward(code).detach()
print(probs)

In [ ]:
classifier.actions

In [ ]:
torch.sum(probs)

## To do

* Complete the method ``train`` of the class ``PolicyGradient``. Observe that a penalty is assigned for illegal actions.
* Test the agent on TicTacToe, against (1) a random adversary and (2) a perfect adversary.<br> What is the first move? Compare with the previous results.
* You now play Connect 4 against a random adversary.<br> What is your expected return after training over 5000 games?

In [ ]:
class PolicyGradient(Agent):
    """Agent learning by policy gradient.
    
    Parameters
    ----------
    environment : object of class Environment
        Environment of the agent.
    player : int
        Player for games (1 or -1, default = default player of the game).
    gamma : float
        Discount rate (in [0, 1], default = 1).
    layers : list of int
        Dimensions of the hidden layers.
    penalty : float
        Penalty for illegal actions (default = -5).
    min_log : float
        Minimal value to compute the log (default = 1e-10)
    """
    def __init__(self, environment, player=None, gamma=1, layers=[10, 5], penalty=-5, min_log=1e-10):
        super(PolicyGradient, self).__init__(environment=environment, player=player)  
        self.nn = Classifier(environment, layers)
        self.action_id = {action: i for i, action in enumerate(self.nn.actions)}
        self.gamma = gamma
        self.penalty = penalty
        self.min_log = min_log
        
    def get_policy(self):
        """Get the current policy."""
        def policy(state):
            actions = self.environment.get_actions(state)
            if len(actions) > 1:
                win_actions = []
                # check win actions
                if self.environment.is_game():
                    next_states = [self.environment.get_next_state(state, action) for action in actions]
                    win_actions = [self.environment.get_reward(next_state) == self.player for next_state in next_states]
                if any(win_actions):
                    probs = np.array(win_actions).astype(float)
                else:
                    # get prob of each action
                    code = self.environment.one_hot_encode(state)
                    code = torch.tensor(code).float()
                    probs = self.nn.forward(code)
                    probs = probs.detach().numpy()
                    # restrict to available actions
                    indices = [self.action_id[action] for action in actions]
                    probs = probs[indices]                    
                # renormalize
                if np.sum(probs) > 0:
                    probs = probs / np.sum(probs)
                else:
                    probs = np.ones(len(actions)) / len(actions)
            else:
                probs = [1]
            return probs, actions
        return policy
    
    def update_policy(self):
        """Update the policy (after training)."""
        self.policy = self.get_policy()
    
    def get_samples(self, state, horizon):
        """Get samples from one episode."""
        rewards = []
        log_probs = []
        log_probs_illegal = []
        self.environment.reset(state)
        state = self.environment.state
        for t in range(horizon):
            action = self.get_action(state)
            if action != "pass":
                i = self.action_id[action]
                code = self.environment.one_hot_encode(state)
                code = torch.tensor(code).float()
                probs = self.nn.forward(code)
                prob = torch.clip(probs[i], self.min_log, 1 - self.min_log)
                log_prob = torch.log(prob).reshape(1)

                actions = self.environment.get_actions(state)
                if action in actions:
                    reward, stop = self.environment.step(action)
                    state = self.environment.state
                    rewards.append(reward)
                    log_probs.append(log_prob)
                else:
                    log_probs_illegal.append(log_prob)
            else:
                reward, stop = self.environment.step(action)
                rewards.append(reward)
                state = self.environment.state
            if stop:
                break
        sample = 0
        for reward in reversed(rewards):
            sample = reward + self.gamma * sample
        return sample, log_probs, log_probs_illegal
        
    def train(self, state=None, horizon=100, n_episodes=1000, batch_size=10, learning_rate=0.01):
        """Train the neural network."""
        optimizer = optim.Adam(self.nn.parameters(), lr=learning_rate)
        loss = 0
        count = 0
        average = 0
        for t in range(n_episodes):
            sample, log_probs, log_probs_illegal = self.get_samples(state, horizon)
            count += 1
            average += (sample - average) / count
            if len(log_probs):
                advantage = sample - average
                loss += - advantage * torch.sum(torch.cat(log_probs))
            if len(log_probs_illegal):
                loss += - self.penalty * torch.sum(torch.cat(log_probs_illegal))
            if not t % batch_size:
                optimizer.zero_grad()
                loss.backward()
                optimizer.step()
                loss = 0

In [ ]:
def eval_agent(agent, n=200):
    returns = agent.get_returns(n_episodes=n)
    vals, counts = np.unique(returns, return_counts=True)
    return dict(zip(vals, counts))

# (1) adversary random
game_r = TicTacToe(adversary_policy='random', player=1, play_first=True)
pg_r = PolicyGradient(game_r, gamma=1, layers=[10,5])
pg_r.train(n_episodes=2000, batch_size=10, learning_rate=0.01) 
pg_r.update_policy()

s0 = game_r.init_state()
first_r = pg_r.get_action(s0)
res_r = eval_agent(pg_r, n=200)

print("PolicyGradient first move vs random:", first_r)
print("Returns counts vs random:", res_r)

# (2) adversary "perfect" = one_step
game_p = TicTacToe(adversary_policy='one_step', player=1, play_first=True)
pg_p = PolicyGradient(game_p, gamma=1, layers=[10,5])
pg_p.train(n_episodes=2000, batch_size=10, learning_rate=0.01)
pg_p.update_policy()

s0 = game_p.init_state()
first_p = pg_p.get_action(s0)
res_p = eval_agent(pg_p, n=200)

print("PolicyGradient first move vs perfect(one_step):", first_p)
print("Returns counts vs perfect(one_step):", res_p)

ANSWER:

PolicyGradient plays (1,1) first against both opponents. Against random it wins most games (184/200). Against one_step it loses more often because the opponent is stronger. Compared to ValueGradient, the first move is still a good opening move (center or corner).

In [ ]:
game_c4 = ConnectFour(adversary_policy='random', player=1, play_first=True)
pg_c4 = PolicyGradient(game_c4, gamma=1, layers=[32, 32]) 
pg_c4.train(n_episodes=5000, batch_size=10, learning_rate=0.005)
pg_c4.update_policy()

returns = pg_c4.get_returns(n_episodes=500)
expected_return = float(np.mean(returns))
vals, counts = np.unique(returns, return_counts=True)

print("Expected return after 5000 games:", expected_return)
print("Return distribution:", dict(zip(vals, counts)))

ANSWER:

After 5000 training games vs a random opponent, the expected return is about 0.852 (mean over 500 games). This corresponds to 463 wins and 37 losses.